# Annotation performance on control proteomes

Compares the sialic acid functions predicted for each control proteome with the functions reported in the literature, for every annotation strategy:

- **DB1–DB6**: DIAMOND blastp against each SialoCat database
- **TM-Vec**: TM-Vec search against DB6 (TM-score ≥ `TMVEC_THRESHOLD`)

Each input table has one row per genome and one column per function, with `OK` when the function is present.

Functions evaluated: Sialylation, Synthesis, Transport, Catabolism and Sialidase.

**Metrics**
- **Precision** = TP / (TP + FP): reliability of the positive predictions (few false positives).
- **Recall** = TP / (TP + FN): ability to recover the true functions (few false negatives).
- **F1-score** = 2TP / (2TP + FP + FN): harmonic mean of precision and recall.
- **Accuracy** = (TP + TN) / (TP + FP + FN + TN).

Only genomes present in the literature table are evaluated.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Parameters

In [ ]:
FUNCTIONS = ["Sialylation", "Synthesis", "Transport", "Catabolism", "Sialidase"]

# Reference: functions reported in the literature for each control genome
TRUTH_FILE = "sialic_functions_literature.tsv"

TMVEC_THRESHOLD = "0.90"

# Predictions of each strategy
PRED_FILES = {
    "DB1": "sialic_functions_DB1.tsv",
    "DB2": "sialic_functions_DB2.tsv",
    "DB3": "sialic_functions_DB3.tsv",
    "DB4": "sialic_functions_DB4.tsv",
    "DB5": "sialic_functions_DB5.tsv",
    "DB6": "sialic_functions_DB6.tsv",
    "TM-Vec": f"sialic_functions_tmvec_{TMVEC_THRESHOLD}.tsv",
}

# Strategy used as reference for the dashed lines in the bar plot
REFERENCE_DB = "DB6"

## Helper functions

In [ ]:
def load_presence(file):
    """Read a genome x function table and convert "OK" to 1, anything else to 0."""
    df = pd.read_csv(file, sep="\t")
    df["genome"] = df["genome"].str.replace(r"_DB.*", "", regex=True)
    df = df.set_index("genome")
    return df.reindex(columns=FUNCTIONS).eq("OK").astype(int)


def confusion(y_true, y_pred):
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    return tp, fp, fn, tn


def scores(tp, fp, fn, tn):
    total = tp + fp + fn + tn
    return {
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "Accuracy": (tp + tn) / total if total > 0 else 0,
        "Precision": tp / (tp + fp) if (tp + fp) > 0 else 0,
        "Recall": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "F1": (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0,
    }


truth = load_presence(TRUTH_FILE)

merged = {}
for db, file in PRED_FILES.items():
    pred = load_presence(file)

    missing = truth.index.difference(pred.index)
    if len(missing) > 0:
        print(f"[WARNING] {db}: genomes from the literature table not found in {file}: {list(missing)}")

    merged[db] = truth.join(pred, lsuffix="_true", rsuffix="_pred")

## 1. Overall performance per strategy

TP, FP, FN and TN are summed over all genomes and functions.

In [ ]:
metrics = []

for db, m in merged.items():
    tp = fp = fn = tn = 0
    for func in FUNCTIONS:
        c = confusion(m[f"{func}_true"], m[f"{func}_pred"])
        tp, fp, fn, tn = tp + c[0], fp + c[1], fn + c[2], tn + c[3]

    metrics.append({"DB": db, **scores(tp, fp, fn, tn)})

metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv("DB_tmvec_performance.tsv", sep="\t", index=False)

metrics_df

In [ ]:
plt.rcParams.update({
    "font.family": "sans-serif",
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
})

bar_metrics = ["Precision", "Recall", "F1"]
colors = ["#4c5a7d", "#2a9d8f", "#8abf69"]
x = np.arange(len(metrics_df["DB"]))
width = 0.25

plt.figure(figsize=(14, 5))

for i, metric in enumerate(bar_metrics):
    plt.bar(x + i * width, metrics_df[metric], width, label=metric, color=colors[i])

# dashed lines at the values of the reference strategy
ref_values = metrics_df[metrics_df["DB"] == REFERENCE_DB].iloc[0]
for i, metric in enumerate(bar_metrics):
    plt.axhline(y=ref_values[metric], linestyle="--", linewidth=1.2,
                color=colors[i], alpha=0.7, zorder=0)

plt.xticks(x + width, metrics_df["DB"])
plt.ylabel("Score")
plt.ylim(0.6, 1.01)
plt.legend(
    loc="upper right",
    bbox_to_anchor=(0.99, 0.98),
    ncol=3,
    borderaxespad=0,
    framealpha=0.9,
    columnspacing=1.2,
    handletextpad=0.5,
)

plt.tight_layout()
plt.savefig("DB_performance_comparison.svg", bbox_inches="tight")
plt.savefig("DB_performance_comparison.png", bbox_inches="tight")
plt.show()

## 2. Performance per strategy and function

In [ ]:
results = []

for db, m in merged.items():
    for func in FUNCTIONS:
        c = confusion(m[f"{func}_true"], m[f"{func}_pred"])
        results.append({"DB": db, "Function": func, **scores(*c)})

f1_df = pd.DataFrame(results)
f1_df.to_csv("F1_per_function_tmvec.tsv", sep="\t", index=False)

f1_df

In [ ]:
heatmaps = {
    "Precision": f1_df.pivot(index="DB", columns="Function", values="Precision"),
    "Recall": f1_df.pivot(index="DB", columns="Function", values="Recall"),
    "F1-score": f1_df.pivot(index="DB", columns="Function", values="F1"),
}

plt.rcParams.update({
    "font.family": "sans-serif",
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 16,
})

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
plt.subplots_adjust(wspace=0.05, right=0.9)   # leave room for the colorbar

# colorbar to the right of the last heatmap, with the same height
pos0 = axes[0].get_position()
pos2 = axes[2].get_position()
cbar_ax = fig.add_axes([pos2.x1 + 0.01, pos0.y0, 0.02, pos0.y1 - pos0.y0])

for i, (title, data) in enumerate(heatmaps.items()):
    last = i == len(heatmaps) - 1
    sns.heatmap(
        data,
        ax=axes[i],
        annot=True,
        cmap="viridis_r",
        vmin=0.3,
        vmax=1,
        cbar=last,
        cbar_ax=cbar_ax if last else None,
        linewidths=0.3,
        linecolor="white",
    )
    axes[i].set_title(title)

for i, ax in enumerate(axes):
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
    if i == 0:
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0, ha="right")
    else:
        ax.set_yticklabels([])   # y labels only on the first heatmap
    ax.set_xlabel("")
    ax.set_ylabel("")

plt.savefig("DB_performance_per_function.svg", bbox_inches="tight")
plt.show()